# Day 15 · PM Session — Database Theory & Relational Algebra
**PG Diploma · AI-ML & Agentic AI Engineering · IIT Gandhinagar**  
**Artifact 7 — Take-Home Assignment (Notebook)**

This notebook covers:
- **Part A** — Pandas operations mapped to relational algebra (with running examples)
- **Part D** — SQL queries on the ride-sharing schema, verified with real output

---

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────
import pandas as pd
import sqlite3
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
print('Libraries loaded ok')

---
## Part A — Sample Data Setup

We create small sample DataFrames that mirror our food delivery schema.  
These are the same tables we designed in the PDF (ER diagram + normalisation).

In [ ]:
# ── Sample DataFrames — Food Delivery Schema ───────────────────────────
df_customers = pd.DataFrame({
    'customer_id': [1, 2, 3, 4],
    'name':        ['Riya Shah', 'Dev Patel', 'Ananya Roy', 'Karan Mehta'],
    'phone':       ['9876543210', '9812345678', '9911223344', '9988776655'],
    'city':        ['Ahmedabad', 'Surat', 'Ahmedabad', 'Baroda'],
})

df_restaurants = pd.DataFrame({
    'restaurant_id': [101, 102, 103],
    'name':          ['Burger Palace', 'Pizza Hub', 'Dosa Corner'],
    'city':          ['Ahmedabad', 'Surat', 'Baroda'],
    'rating':        [4.2, 4.5, 3.9],
})

df_agents = pd.DataFrame({
    'agent_id':    [201, 202, 203],
    'name':        ['Arjun K', 'Priya S', 'Rahul D'],
    'vehicle_type':['Bike', 'Scooter', 'Bike'],
    'rating':      [4.8, 4.6, 4.3],
})

df_orders = pd.DataFrame({
    'order_id':      [5001, 5002, 5003, 5004, 5005],
    'customer_id':   [1,    2,    1,    3,    4   ],
    'restaurant_id': [101,  102,  102,  101,  103 ],
    'agent_id':      [201,  202,  201,  203,  202 ],
    'status':        ['delivered', 'delivered', 'cancelled', 'delivered', 'delivered'],
    'total_amount':  [290, 200, 150, 450, 320],
})

df_menu_items = pd.DataFrame({
    'menu_item_id': [301, 302, 303, 304, 305],
    'restaurant_id':[101, 101, 102, 102, 103],
    'name':         ['Cheeseburger', 'Fries', 'Margherita', 'Pepperoni', 'Masala Dosa'],
    'price':        [120, 50, 200, 250, 90],
    'category':     ['Main', 'Side', 'Main', 'Main', 'Main'],
})

df_order_items = pd.DataFrame({
    'item_id':      [1, 2, 3, 4, 5, 6],
    'order_id':     [5001, 5001, 5002, 5003, 5004, 5005],
    'menu_item_id': [301, 302, 303, 301, 304, 305],
    'qty':          [2, 1, 1, 1, 1, 2],
    'price':        [120, 50, 200, 120, 250, 90],
})

print('Sample data ready')
print(f'  Customers:   {len(df_customers)} rows')
print(f'  Restaurants: {len(df_restaurants)} rows')
print(f'  Orders:      {len(df_orders)} rows')
print(f'  Order Items: {len(df_order_items)} rows')
print(f'  Menu Items:  {len(df_menu_items)} rows')

---
## Part A4 — Pandas Operations Mapped to Relational Algebra

### Mapping 1: `merge` → Natural Join (⋈)

In [ ]:
# ── Mapping 1: merge → Natural Join ───────────────────────────────────
# Relational Algebra: Order ⋈ Customer
# SQL equivalent:     SELECT * FROM orders JOIN customers USING (customer_id)

print('=== Pandas merge (Natural Join) ===')
print('RA notation: π order_id, O.customer_id, name, status, total_amount (Order ⋈ Customer)\n')

# The operation
result_join = df_orders.merge(df_customers, on='customer_id')[[
    'order_id', 'customer_id', 'name', 'status', 'total_amount'
]]

print(result_join.to_string(index=False))
print(f'\nRows returned: {len(result_join)}')

### Mapping 2: `groupby + agg` → Grouping Operator (γ)

In [ ]:
# ── Mapping 2: groupby + agg → Grouping Operator ──────────────────────
# Relational Algebra: γ restaurant_id; SUM(total_amount), COUNT(order_id) (Order)
# SQL equivalent:     SELECT restaurant_id, SUM(total_amount), COUNT(*) FROM orders GROUP BY restaurant_id

print('=== Pandas groupby + agg (Grouping / Aggregation) ===')
print('RA notation: γ restaurant_id ; SUM(total_amount), COUNT(order_id) (σ status=delivered (Order))\n')

# Only count delivered orders
delivered = df_orders[df_orders['status'] == 'delivered']

result_agg = (
    delivered
    .groupby('restaurant_id')
    .agg(
        total_revenue = ('total_amount', 'sum'),
        num_orders    = ('order_id', 'count'),
        avg_order_val = ('total_amount', 'mean'),
    )
    .reset_index()
    .merge(df_restaurants[['restaurant_id', 'name']], on='restaurant_id')
    [['name', 'num_orders', 'total_revenue', 'avg_order_val']]
)

print(result_agg.to_string(index=False))

### Mapping 3: Boolean Filter → Selection (σ)

In [ ]:
# ── Mapping 3: Boolean filter → Selection ─────────────────────────────
# Relational Algebra: σ status = 'delivered' (Order)
# SQL equivalent:     SELECT * FROM orders WHERE status = 'delivered'

print("=== Pandas boolean filter (Selection / σ) ===")
print("RA notation: σ status = 'delivered' (Order)\n")

result_filter = df_orders[df_orders['status'] == 'delivered'].reset_index(drop=True)
print(result_filter.to_string(index=False))
print(f'\nRows after filter: {len(result_filter)} out of {len(df_orders)}')

# Combining selection + projection (very common in real code)
print("\n--- Combined: σ + π (filter then pick columns) ---")
print("RA: π order_id, customer_id, total_amount (σ status='delivered' AND total_amount > 200 (Order))\n")
result_combined = (
    df_orders[
        (df_orders['status'] == 'delivered') &
        (df_orders['total_amount'] > 200)
    ][['order_id', 'customer_id', 'total_amount']]
    .reset_index(drop=True)
)
print(result_combined.to_string(index=False))

---
## Part D — Ride-Sharing Schema: SQL Queries via SQLite

We create an in-memory SQLite database with the ride-sharing schema proposed by the AI,  
then run and verify queries — including window functions.

In [ ]:
# ── Build ride-sharing DB in SQLite ────────────────────────────────────
conn = sqlite3.connect(':memory:')
cur  = conn.cursor()

# Tables
cur.executescript("""
CREATE TABLE User (
    user_id   INTEGER PRIMARY KEY,
    name      TEXT NOT NULL,
    phone     TEXT,
    email     TEXT,
    role      TEXT CHECK(role IN ('rider','driver'))
);

CREATE TABLE Vehicle (
    vehicle_id   INTEGER PRIMARY KEY,
    driver_id    INTEGER REFERENCES User(user_id),
    type         TEXT,
    plate_number TEXT,
    model        TEXT
);

CREATE TABLE Ride (
    ride_id          INTEGER PRIMARY KEY,
    rider_id         INTEGER REFERENCES User(user_id),
    driver_id        INTEGER REFERENCES User(user_id),
    vehicle_id       INTEGER REFERENCES Vehicle(vehicle_id),
    pickup_location  TEXT,
    drop_location    TEXT,
    fare             REAL,
    status           TEXT,
    start_time       TEXT,
    end_time         TEXT
);

CREATE TABLE Payment (
    payment_id   INTEGER PRIMARY KEY,
    ride_id      INTEGER REFERENCES Ride(ride_id),
    amount       REAL,
    method       TEXT,
    payment_time TEXT,
    status       TEXT
);

CREATE TABLE Rating (
    rating_id  INTEGER PRIMARY KEY,
    ride_id    INTEGER REFERENCES Ride(ride_id),
    rated_by   INTEGER REFERENCES User(user_id),
    rated_user INTEGER REFERENCES User(user_id),
    score      INTEGER CHECK(score BETWEEN 1 AND 5),
    comments   TEXT
);
""")

# Sample data
users = [
    (1,'Riya Shah',   '9876543210','riya@mail.com',  'rider'),
    (2,'Dev Patel',   '9812345678','dev@mail.com',   'rider'),
    (3,'Ananya Roy',  '9911223344','ananya@mail.com','rider'),
    (4,'Arjun Kumar', '9001122334','arjun@mail.com', 'driver'),
    (5,'Priya Singh', '9112233445','priya@mail.com', 'driver'),
    (6,'Rahul Desai', '9223344556','rahul@mail.com', 'driver'),
]
vehicles = [
    (101, 4, 'Sedan',  'GJ01AB1234', 'Maruti Swift'),
    (102, 5, 'SUV',    'GJ01CD5678', 'Mahindra XUV'),
    (103, 6, 'Hatchback','GJ02EF9012','Hyundai i20'),
]
rides = [
    (1001, 1, 4, 101, 'Satellite', 'CG Road',    180.0, 'completed', '2024-03-01 09:00','2024-03-01 09:25'),
    (1002, 2, 5, 102, 'Bodakdev',  'Vastrapur',  220.0, 'completed', '2024-03-01 10:00','2024-03-01 10:30'),
    (1003, 1, 6, 103, 'Navrangpura','Ellis Bridge',95.0, 'completed', '2024-03-02 08:00','2024-03-02 08:15'),
    (1004, 3, 4, 101, 'Gota',      'Chandkheda', 310.0, 'completed', '2024-03-02 11:00','2024-03-02 11:40'),
    (1005, 2, 6, 103, 'Bopal',     'SG Highway', 145.0, 'cancelled', '2024-03-03 07:00', None),
    (1006, 1, 5, 102, 'Prahlad Nagar','Ambawadi',260.0, 'completed', '2024-03-03 14:00','2024-03-03 14:35'),
]
payments = [
    (201, 1001, 180.0, 'UPI',   '2024-03-01 09:26', 'success'),
    (202, 1002, 220.0, 'Card',  '2024-03-01 10:31', 'success'),
    (203, 1003,  95.0, 'Cash',  '2024-03-02 08:16', 'success'),
    (204, 1004, 310.0, 'UPI',   '2024-03-02 11:41', 'success'),
    (205, 1006, 260.0, 'Card',  '2024-03-03 14:36', 'success'),
]
ratings = [
    (301, 1001, 1, 4, 5, 'Smooth ride, very punctual'),
    (302, 1002, 2, 5, 4, 'Good driver, clean car'),
    (303, 1003, 1, 6, 3, 'Took a longer route'),
    (304, 1004, 3, 4, 5, 'Excellent, would recommend'),
    (305, 1006, 1, 5, 5, 'Great experience'),
]

cur.executemany('INSERT INTO User VALUES (?,?,?,?,?)', users)
cur.executemany('INSERT INTO Vehicle VALUES (?,?,?,?,?)', vehicles)
cur.executemany('INSERT INTO Ride VALUES (?,?,?,?,?,?,?,?,?,?)', rides)
cur.executemany('INSERT INTO Payment VALUES (?,?,?,?,?,?)', payments)
cur.executemany('INSERT INTO Rating VALUES (?,?,?,?,?,?)', ratings)
conn.commit()
print('Ride-sharing DB created and populated in SQLite')
print(f'  Users: {cur.execute("SELECT COUNT(*) FROM User").fetchone()[0]}')
print(f'  Rides: {cur.execute("SELECT COUNT(*) FROM Ride").fetchone()[0]}')
print(f'  Payments: {cur.execute("SELECT COUNT(*) FROM Payment").fetchone()[0]}')

### D-SQL1: Total earnings per driver with their average rating
*(Window function: RANK to rank drivers by earnings)*

In [ ]:
# SQL Query 1 — Driver earnings + avg rating + rank by earnings
# Window function: RANK() OVER (ORDER BY total_earned DESC)

q1 = """
SELECT
    u.name                                          AS driver_name,
    COUNT(r.ride_id)                                AS total_rides,
    ROUND(SUM(r.fare), 2)                           AS total_earned,
    ROUND(AVG(rat.score), 2)                        AS avg_rating,
    RANK() OVER (ORDER BY SUM(r.fare) DESC)         AS earnings_rank
FROM User u
JOIN Ride r   ON u.user_id = r.driver_id  AND r.status = 'completed'
LEFT JOIN Rating rat ON r.ride_id = rat.ride_id AND rat.rated_user = u.user_id
WHERE u.role = 'driver'
GROUP BY u.user_id, u.name
ORDER BY total_earned DESC;
"""

print('Query 1 — Driver earnings & ratings (with RANK window function)')
print('='*65)
result1 = pd.read_sql_query(q1, conn)
print(result1.to_string(index=False))

### D-SQL2: Running total of fare per rider (cumulative spend)
*(Window function: SUM() OVER with PARTITION BY and ORDER BY)*

In [ ]:
# SQL Query 2 — Running cumulative fare per rider over time
# Window function: SUM(fare) OVER (PARTITION BY rider_id ORDER BY start_time)

q2 = """
SELECT
    u.name                                                   AS rider_name,
    r.ride_id,
    r.start_time,
    r.pickup_location,
    r.drop_location,
    r.fare,
    SUM(r.fare) OVER (
        PARTITION BY r.rider_id
        ORDER BY r.start_time
    )                                                        AS cumulative_spend
FROM Ride r
JOIN User u ON r.rider_id = u.user_id
WHERE r.status = 'completed'
ORDER BY rider_name, r.start_time;
"""

print('Query 2 — Cumulative fare per rider (SUM window function)')
print('='*65)
result2 = pd.read_sql_query(q2, conn)
print(result2.to_string(index=False))

In [ ]:
# SQL Query 3 — Payment method breakdown: count and percentage
q3 = """
SELECT
    method,
    COUNT(*)                                              AS num_payments,
    ROUND(SUM(amount), 2)                                 AS total_amount,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1)   AS pct_of_transactions
FROM Payment
WHERE status = 'success'
GROUP BY method
ORDER BY total_amount DESC;
"""

print('Query 3 — Payment method breakdown (% share using window function)')
print('='*65)
result3 = pd.read_sql_query(q3, conn)
print(result3.to_string(index=False))

In [ ]:
# SQL Query 4 — Riders who spent more than the average rider spend
q4 = """
SELECT
    u.name                     AS rider_name,
    ROUND(SUM(r.fare), 2)      AS total_spent,
    ROUND(AVG(SUM(r.fare)) OVER (), 2)  AS avg_spend_all_riders
FROM User u
JOIN Ride r ON u.user_id = r.rider_id AND r.status = 'completed'
GROUP BY u.user_id, u.name
HAVING SUM(r.fare) > (
    SELECT AVG(total) FROM (
        SELECT SUM(fare) AS total
        FROM Ride
        WHERE status = 'completed'
        GROUP BY rider_id
    )
)
ORDER BY total_spent DESC;
"""

print('Query 4 — Riders who spent above average')
print('='*65)
result4 = pd.read_sql_query(q4, conn)
print(result4.to_string(index=False))

In [ ]:
# SQL Query 5 — Lag: compare each ride fare to the previous ride fare for same rider
# Window function: LAG() to look at the previous row within the same rider

q5 = """
SELECT
    u.name                  AS rider_name,
    r.ride_id,
    r.start_time,
    r.fare,
    LAG(r.fare) OVER (
        PARTITION BY r.rider_id
        ORDER BY r.start_time
    )                       AS prev_ride_fare,
    ROUND(
        r.fare - LAG(r.fare) OVER (
            PARTITION BY r.rider_id
            ORDER BY r.start_time
        ), 2
    )                       AS fare_change
FROM Ride r
JOIN User u ON r.rider_id = u.user_id
WHERE r.status = 'completed'
ORDER BY rider_name, r.start_time;
"""

print('Query 5 — Fare change from previous ride (LAG window function)')
print('='*65)
result5 = pd.read_sql_query(q5, conn)
print(result5.to_string(index=False))

---
## Part D — 3NF Evaluation of AI-Generated Schema

The AI proposed this schema for a ride-sharing app:  
`User`, `Vehicle`, `Ride`, `Payment`, `Rating`

Let's verify each table for 3NF violations.

In [ ]:
# 3NF Check — print a structured evaluation
nf_check = [
    {
        'table': 'User',
        'pk': 'user_id',
        'non_key_cols': ['name', 'phone', 'email', 'role'],
        'verdict': '3NF OK',
        'reason': 'All columns depend directly on user_id. No partial or transitive dependencies.'
    },
    {
        'table': 'Vehicle',
        'pk': 'vehicle_id',
        'non_key_cols': ['driver_id', 'type', 'plate_number', 'model'],
        'verdict': '3NF OK',
        'reason': 'All columns depend on vehicle_id. driver_id is a FK, not a transitive dependency.'
    },
    {
        'table': 'Ride',
        'pk': 'ride_id',
        'non_key_cols': ['rider_id', 'driver_id', 'vehicle_id', 'pickup_location',
                         'drop_location', 'fare', 'status', 'start_time', 'end_time'],
        'verdict': '3NF OK',
        'reason': 'All attributes describe this specific ride. No non-key column determines another non-key column.'
    },
    {
        'table': 'Payment',
        'pk': 'payment_id',
        'non_key_cols': ['ride_id', 'amount', 'method', 'payment_time', 'status'],
        'verdict': '3NF OK',
        'reason': 'All columns depend on payment_id. amount and ride_id are independent facts.'
    },
    {
        'table': 'Rating',
        'pk': 'rating_id',
        'non_key_cols': ['ride_id', 'rated_by', 'rated_user', 'score', 'comments'],
        'verdict': '3NF OK',
        'reason': 'All columns directly describe this rating instance. No transitive dependency found.'
    },
]

print('3NF Evaluation — AI-Generated Ride-Sharing Schema')
print('='*70)
for t in nf_check:
    print(f"\nTable: {t['table']}")
    print(f"  PK: {t['pk']}")
    print(f"  Verdict: {t['verdict']}")
    print(f"  Reason:  {t['reason']}")

print()
print('Missing relationships / design issues found:')
issues = [
    "1. role column in User ('rider'/'driver') — better as a separate Driver table to store vehicle_type, licence, etc.",
    "2. No SurgePricing table — a real app needs to record the multiplier applied at time of booking.",
    "3. No explicit ActiveRide link — hard to query 'which driver is currently on a ride' without a status filter + join.",
    "4. Ride.status has no constraint table — magic strings like 'completed', 'cancelled' should be in a lookup table.",
]
for issue in issues:
    print(f'  {issue}')

---
## Quick Summary: What We Did

| Part | What | Tool |
|---|---|---|
| A1 | ER Diagram (6 entities, all cardinalities) | In PDF |
| A2 | Normalisation UNF → 1NF → 2NF → 3NF | In PDF |
| A3 | 5 Relational Algebra expressions | In PDF |
| A4 | 3 Pandas ops mapped to RA | This notebook |
| B  | BCNF violation + decomposition | In PDF |
| C1 | Update anomaly with e-commerce example | In PDF |
| C2 | Price + PriceHistory schema in 3NF | In PDF |
| C3 | ACID: Isolation at risk in double-booking | In PDF |
| D  | Ride-sharing schema + 5 SQL window queries | This notebook |

In [ ]:
conn.close()
print('Done — all queries verified successfully.')